# Pipeline de XGBoost con Target Encoding

Este cuaderno implementa un pipeline de modelado predictivo para el precio de vehículos usados utilizando **XGBoost** y **Target Encoding** para variables categóricas.

## 1. Importación de Librerías

Cargamos las librerías necesarias de análisis de datos y scikit-learn. También instalamos automáticamente las dependencias externas `category_encoders` y `xgboost` si no están presentes.

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Intentamos importar category_encoders, si no existe lo instalamos automáticamente
try:
    from category_encoders import TargetEncoder
except ImportError:
    print("Instalando la librería 'category_encoders' necesaria...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "category_encoders"])
    from category_encoders import TargetEncoder

# Intentamos importar xgboost, si no existe lo instalamos automáticamente
try:
    import xgboost as xgb
except ImportError:
    print("Instalando la librería 'xgboost' necesaria...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    import xgboost as xgb

## 2. Cargar Datos

Cargamos el archivo `vehicles.csv`. Si no está disponible localmente, se intentará descargar desde Kaggle utilizando las credenciales provistas.

In [ ]:
if os.path.exists('vehicles.csv'):
    print("Cargando datos desde 'vehicles.csv'...")
    df = pd.read_csv('vehicles.csv', encoding='latin1', on_bad_lines='skip', low_memory=False)
else:
    print("El archivo 'vehicles.csv' no existe localmente. Intentando descargar de Kaggle...")
    try:
        os.environ['KAGGLE_API_TOKEN'] = "KGAT_a0fcdeed3125aab3d652c9bc808ee163"
        import kaggle
        kaggle.api.dataset_download_files('austinreese/craigslist-carstrucks-data', path='.', unzip=True)
        print("¡Descarga completada con éxito!")
        df = pd.read_csv('vehicles.csv', encoding='latin1', on_bad_lines='skip', low_memory=False)
    except Exception as e:
        print(f"Error al descargar de Kaggle: {e}")
        print("Por favor, coloca 'vehicles.csv' en este directorio y vuelve a intentarlo.")
        raise FileNotFoundError("vehicles.csv no encontrado.")

## 3. Limpieza y Filtrado de Datos

Eliminamos filas duplicadas y columnas que no aportan información predictiva significativa (como URLs, descripciones, coordenadas, etc.). Aplicamos también filtros comerciales para acotar el rango de precios, el odómetro y los años del vehículo según las conclusiones del EDA.

In [ ]:
print("Preprocesando y limpiando el dataset...")
df_clean = df.drop_duplicates()
cols_irrelevantes = ['id', 'url', 'region_url', 'image_url', 'description',
                     'VIN', 'county', 'posting_date', 'lat', 'long']
df_clean = df_clean.drop(columns=[c for c in cols_irrelevantes if c in df_clean.columns])

# Filtrado comercial
df_clean = df_clean[(df_clean['price'] > 500) & (df_clean['price'] <= 150000)]
df_clean = df_clean[df_clean['odometer'].isna() | ((df_clean['odometer'] >= 1) & (df_clean['odometer'] <= 500000))]
df_clean = df_clean[df_clean['year'].isna() | ((df_clean['year'] >= 1980) & (df_clean['year'] <= 2027))]

## 4. División de Conjuntos (Entrenamiento y Prueba)

Separamos la variable objetivo `price` del resto de las características y dividimos en conjuntos de entrenamiento (`train`) y prueba (`test`).

In [ ]:
X = df_clean.drop('price', axis=1)
y = df_clean['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identificación automática de tipos de columnas
columnas_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
columnas_categoricas = X_train.select_dtypes(include=['object', 'category']).columns

## 5. Diseño del Pipeline de Preprocesamiento

- **Numéricas**: Imputación de nulos por la mediana y estandarización.
- **Categóricas**: Imputación de nulos como 'unknown' y aplicación de `TargetEncoder` con suavizado (`smoothing=10`) para evitar sobreajuste en categorías poco frecuentes.

In [ ]:
# Pipeline numérico
pipeline_numerico = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

# Pipeline categórico con Target Encoding
pipeline_cat_alta = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('target_encoder', TargetEncoder(smoothing=10))
])

# Ensamblado del preprocesador
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipeline_numerico, columnas_numericas),
        ('cat', pipeline_cat_alta, columnas_categoricas)
    ])

## 6. Pipeline de Predicción con XGBoost

Ensamblamos el ColumnTransformer con el regresor `XGBRegressor`.

In [ ]:
pipeline_xgb = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=7,
        random_state=42,
        n_jobs=-1
    ))
])

## 7. Entrenamiento del Modelo

Entrenamos el pipeline completo en el conjunto de entrenamiento.

In [ ]:
print("Limpiando nulos, aplicando Target Encoding y entrenando XGBoost...")
pipeline_xgb.fit(X_train, y_train)
print("¡Entrenamiento completado sin errores!")